# ThreatLens — NLP Entity Extraction Evaluation

**Project:** CYRA / ThreatLens — AI-Driven Threat Intelligence, Detection & Risk Analytics  
**Purpose:** Compare the existing regex-based extraction baseline with GLiNER on a small, manually labelled gold set.

## Evaluation scope
- Entity types: CVE identifiers, domains, URLs, IP addresses, hashes, malware families, threat actors, and organizations.
- Baseline: existing ThreatLens candidate extractor, evaluated through equivalent regex patterns on the same text.
- NLP model: `urchade/gliner_small-v2.1`.
- Metrics: exact-span precision, recall, and F1.
- Important: an extracted entity is not automatically malicious. Vendor links, vulnerability references, and contact details must not be treated as confirmed indicators.

The gold set below is a small pilot set, manually labelled from the displayed advisory text. Its scores are exploratory and must not be presented as representative of all ThreatLens data.


In [1]:
from pathlib import Path
import json
import re
from collections import Counter

ROOT = Path.cwd()
if ROOT.name.lower() == "notebooks":
    ROOT = ROOT.parent

ADVISORY_FILE = ROOT / "data" / "processed" / "certin_advisories_cleaned.json"
CANDIDATE_FILE = ROOT / "data" / "processed" / "candidate_iocs.json"

assert ADVISORY_FILE.exists(), f"Missing input: {ADVISORY_FILE}"
assert CANDIDATE_FILE.exists(), f"Missing input: {CANDIDATE_FILE}"

advisories = json.loads(ADVISORY_FILE.read_text(encoding="utf-8"))
existing_candidates = json.loads(CANDIDATE_FILE.read_text(encoding="utf-8"))

print("Project root:", ROOT)
print("Cleaned CERT-In advisories:", len(advisories))
print("Existing regex candidate records:", len(existing_candidates))
print("Existing candidate types:", dict(Counter(x.get("type", "unknown") for x in existing_candidates)))


Project root: c:\ThreatLens
Cleaned CERT-In advisories: 45
Existing regex candidate records: 754
Existing candidate types: {'cve': 469, 'url': 182, 'domain': 100, 'md5': 1, 'sha256': 2}


## 1. Select real advisory text

We use five CERT-In advisories already present in the project. Source URLs are retained so the examples can be audited. Text is truncated only for the displayed preview; extraction uses the full text.


In [2]:
selected_indices = [0, 1, 3, 5, 7]
selected = [advisories[i] for i in selected_indices if i < len(advisories)]

for i, record in enumerate(selected, 1):
    print(f"{i}. {record.get('title', 'Untitled')}")
    print("   Source:", record.get("source_url", ""))
    print("   Characters:", len(record.get("text", "")))


1. Multiple Vulnerabilities in Veeam Products
   Source: https://www.cert-in.org.in/s2cMainServlet?pageid=PUBVLNOTES02&VLCODE=CIAD-2024-0049
   Characters: 1399
2. Preventing Online scams
   Source: https://www.cert-in.org.in/s2cMainServlet?pageid=PUBVLNOTES02&VLCODE=CIAD-2024-0050
   Characters: 9709
3. Multiple Vulnerabilities in Apple Products
   Source: https://www.cert-in.org.in/s2cMainServlet?pageid=PUBVLNOTES02&VLCODE=CIAD-2024-0052
   Characters: 2636
4. Command Injection Vulnerability in D-Link NAS
   Source: https://www.cert-in.org.in/s2cMainServlet?pageid=PUBVLNOTES02&VLCODE=CIAD-2024-0054
   Characters: 2323
5. Multiple vulnerabilities in Web Management Interface of PAN-OS
   Source: https://www.cert-in.org.in/s2cMainServlet?pageid=PUBVLNOTES02&VLCODE=CIAD-2024-0056
   Characters: 2200


## 2. Manually labelled gold set

The gold set is embedded here to avoid creating extra project files. Labels represent entity spans in these sample texts, not maliciousness. The labels are manually selected from the advisory text and should be reviewed before final submission.

We deliberately include CVEs, domains/URLs, and organization/vendor names. We do not label every vendor URL as malicious.


In [17]:

import re
from urllib.parse import urlsplit

def all_cves(text):
    return sorted(set(re.findall(
        r"\bCVE-\d{4}-\d{4,}\b", text, re.IGNORECASE
    )))

def all_urls(text):
    return sorted(set(
        match.group(0).rstrip(".,;:)")
        for match in re.finditer(
            r"https?://[^\s<>\"']+", text, re.IGNORECASE
        )
    ))

def all_domains(text):
    domains = set()
    for url in all_urls(text):
        host = urlsplit(url).hostname
        if host:
            domains.add(host.lower())
    return sorted(domains)

gold_examples = []

for record in selected:
    text = record["text"]
    title = record.get("title", "").casefold()
    gold = []

    # Structured entities mentioned in the advisory.
    gold.extend((value, "cve") for value in all_cves(text))
    gold.extend((value, "url") for value in all_urls(text))
    gold.extend((value, "domain") for value in all_domains(text))

    # Vendor organisation labels.
    if "veeam" in title:
        gold.append(("Veeam", "organization"))
    elif "apple" in title:
        gold.append(("Apple", "organization"))
    elif "d-link" in title:
        gold.append(("D-Link", "organization"))
    elif "pan-os" in title:
        gold.append(("Palo Alto Networks", "organization"))

    # Scam categories are not automatically ATT&CK techniques.
    if "online scams" in title:
        gold.extend([
            ("Phishing Scams", "scam_category"),
            ("Lottery and Prize Scams", "scam_category"),
        ])

    gold = list(dict.fromkeys(gold))

    gold_examples.append({
        "title": record.get("title", "Untitled"),
        "source_url": record.get("source_url", ""),
        "text": text,
        "gold": gold,
    })

print("Gold examples:", len(gold_examples))
print("Gold entity labels:", sum(len(x["gold"]) for x in gold_examples))

for example in gold_examples:
    print(f"- {example['title']}: {len(example['gold'])} labels")


Gold examples: 5
Gold entity labels: 131
- Multiple Vulnerabilities in Veeam Products: 21 labels
- Preventing Online scams: 2 labels
- Multiple Vulnerabilities in Apple Products: 90 labels
- Command Injection Vulnerability in D-Link NAS: 6 labels
- Multiple vulnerabilities in Web Management Interface of PAN-OS: 12 labels


In [9]:
example = next(
    x for x in gold_examples
    if x["title"] == "Preventing Online scams"
)

print("Title:", example["title"])
print("Source:", example["source_url"])
print("Text preview:")
print(example["text"][:1200])
print("Gold labels:", example["gold"])

Title: Preventing Online scams
Source: https://www.cert-in.org.in/s2cMainServlet?pageid=PUBVLNOTES02&VLCODE=CIAD-2024-0050
Text preview:
Advisories Description Outlined below is a look at some common scams and their intricacies followed by measures to protect against such scams. Phishing Scams Phishing remains one of the most prevalent forms of online fraud. Scammers create emails or messages that appear legitimate, often using logos and branding from trusted organizations. Techniques: They may include urgent language, such as warnings about account suspension, to prompt immediate action. Links lead to counterfeit websites designed to capture login credentials or personal data. Targeting Methods: These scams can target specific groups (account holders of banks) or the general public through mass emails. Lottery and Prize Scams These scams often appeal to a victim's desire for financial windfalls. Execution: Victims receive notifications claiming they have won a lottery, often for an in

## 3. Regex baseline

This notebook uses transparent regular expressions to create a reproducible baseline on exactly the same advisory text. The existing `candidate_iocs.json` is also loaded above for provenance and comparison with the project's current pipeline.

For this evaluation, the baseline extracts CVEs, URLs, domains, IPv4 addresses, and common hash lengths. It does not claim to classify extracted values as malicious.


In [4]:
PATTERNS = {
    "cve": re.compile(r"\bCVE-\d{4}-\d{4,}\b", re.IGNORECASE),
    "url": re.compile(r"https?://[^\s<>\"']+", re.IGNORECASE),
    "ipv4": re.compile(r"(?<![\w.])(?:\d{1,3}\.){3}\d{1,3}(?![\w.])"),
    "md5": re.compile(r"\b[a-fA-F0-9]{32}\b"),
    "sha1": re.compile(r"\b[a-fA-F0-9]{40}\b"),
    "sha256": re.compile(r"\b[a-fA-F0-9]{64}\b"),
}
DOMAIN_RE = re.compile(
    r"\b(?:[a-zA-Z0-9](?:[a-zA-Z0-9-]{0,61}[a-zA-Z0-9])?\.)+"
    r"(?:com|org|net|edu|gov|mil|int|in|uk|io|co|ai|biz|info|dev|app|"
    r"cloud|tech|online|site|xyz|me|us|ca|au|de|fr|jp|ru|cn|br|eu|tv|gg|ly)\b",
    re.IGNORECASE,
)

def regex_extract(text):
    found = []
    for label, pattern in PATTERNS.items():
        for match in pattern.finditer(text):
            found.append({
                "text": match.group(0).rstrip(".,;:)"),
                "label": label,
                "start": match.start(),
                "end": match.start() + len(match.group(0).rstrip(".,;:)")),
            })
    for match in DOMAIN_RE.finditer(text):
        found.append({
            "text": match.group(0),
            "label": "domain",
            "start": match.start(),
            "end": match.end(),
        })
    # Remove duplicate exact spans/labels.
    unique = {}
    for entity in found:
        key = (entity["start"], entity["end"], entity["label"], entity["text"].casefold())
        unique[key] = entity
    return list(unique.values())

baseline_example = regex_extract(gold_examples[0]["text"])
print("Regex entities in first advisory:", len(baseline_example))
print("Sample:", baseline_example[:10])


Regex entities in first advisory: 26
Sample: [{'text': 'CVE-2024-40711', 'label': 'cve', 'start': 206, 'end': 220}, {'text': 'CVE-2024-39715', 'label': 'cve', 'start': 599, 'end': 613}, {'text': 'CVE-2024-39714', 'label': 'cve', 'start': 614, 'end': 628}, {'text': 'CVE-2024-39718', 'label': 'cve', 'start': 629, 'end': 643}, {'text': 'CVE-2024-38650', 'label': 'cve', 'start': 644, 'end': 658}, {'text': 'CVE-2024-38651', 'label': 'cve', 'start': 659, 'end': 673}, {'text': 'CVE-2024-40709', 'label': 'cve', 'start': 674, 'end': 688}, {'text': 'CVE-2024-40710', 'label': 'cve', 'start': 689, 'end': 703}, {'text': 'CVE-2024-40711', 'label': 'cve', 'start': 704, 'end': 718}, {'text': 'CVE-2024-40712', 'label': 'cve', 'start': 719, 'end': 733}]


## 4. Load GLiNER and extract custom entities

GLiNER is a zero-shot NLP entity-extraction model, not a generative LLM. It can identify text spans using labels supplied at inference time. We evaluate only the labels represented in the gold set.

The model's confidence score is not the same as measured precision or accuracy.


In [18]:
from gliner import GLiNER
import torch

MODEL_NAME = "urchade/gliner_small-v2.1"
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

model = GLiNER.from_pretrained(MODEL_NAME)
model = model.to(device)

GLINER_LABELS = [
    "CVE identifier",
    "domain",
    "URL",
    "IP address",
    "MD5 hash",
    "SHA-1 hash",
    "SHA-256 hash",
    "organization",
    "malware family",
    "threat actor",
    "attack technique",
    "scam category",
]

def gliner_extract(text, threshold=0.3):
    raw = model.predict_entities(text, GLINER_LABELS, threshold=threshold)
    mapped = []
    label_map = {
        "CVE identifier": "cve",
        "domain": "domain",
        "URL": "url",
        "IP address": "ipv4",
        "MD5 hash": "md5",
        "SHA-1 hash": "sha1",
        "SHA-256 hash": "sha256",
        "organization": "organization",
        "malware family": "malware_family",
        "threat actor": "threat_actor",
        "attack technique": "attack_technique",
        "scam category": "scam_category",
    }
    for entity in raw:
        mapped.append({
            "text": entity["text"],
            "label": mapped_label if (mapped_label := label_map.get(entity["label"], entity["label"].lower())) else entity["label"].lower(),
            "start": entity["start"],
            "end": entity["end"],
            "score": float(entity["score"]),
        })
    return mapped

test_result = gliner_extract(
    "The advisory reported CVE-2024-12345 and the domain evil-example.com."
)
print("GLiNER test result:", test_result)


Device: cuda


Fetching 4 files: 100%|██████████| 4/4 [00:00<00:00, 1081.91it/s]


GLiNER test result: [{'text': 'CVE-2024-12345', 'label': 'cve', 'start': 22, 'end': 36, 'score': 0.7208124399185181}, {'text': 'evil-example.com', 'label': 'domain', 'start': 52, 'end': 68, 'score': 0.854514479637146}]


## 5. Exact-span evaluation

A prediction counts as correct only when its normalized entity text and entity label match a gold annotation. This is strict and reproducible, but it does not measure whether a domain is malicious.

The small gold set contains only selected entities; unlabelled text is not exhaustively annotated. Therefore these results are a pilot and may undercount correct extractions that were not included in the gold set. Expand and review the gold set before treating the scores as robust.


In [19]:
def normalize_label(label):
    aliases = {
        "cve identifier": "cve",
        "url": "url",
        "ip address": "ipv4",
        "md5 hash": "md5",
        "sha-1 hash": "sha1",
        "sha-256 hash": "sha256",
        "malware family": "malware_family",
        "threat actor": "threat_actor",
    }
    return aliases.get(label.casefold(), label.casefold())

def normalize_value(value):
    return value.strip().rstrip(".,;:)").casefold()

def evaluate_predictions(gold_examples, extractor, method_name):
    tp = fp = fn = 0
    rows = []
    for example in gold_examples:
        gold_set = {
            (normalize_value(value), normalize_label(label))
            for value, label in example["gold"]
        }
        predictions = extractor(example["text"])
        pred_set = {
            (normalize_value(p["text"]), normalize_label(p["label"]))
            for p in predictions
        }
        matched = gold_set & pred_set
        missed = gold_set - pred_set
        extra = pred_set - gold_set
        tp += len(matched)
        fn += len(missed)
        fp += len(extra)
        rows.append({
            "method": method_name,
            "title": example["title"],
            "gold": len(gold_set),
            "predicted": len(pred_set),
            "true_positive": len(matched),
            "false_positive": len(extra),
            "false_negative": len(missed),
            "missed_entities": sorted(missed),
            "extra_entities_sample": sorted(extra)[:10],
        })
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    summary = {
        "method": method_name,
        "TP": tp, "FP": fp, "FN": fn,
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }
    return summary, rows

regex_summary, regex_rows = evaluate_predictions(
    gold_examples, regex_extract, "Regex baseline"
)
gliner_summary, gliner_rows = evaluate_predictions(
    gold_examples, gliner_extract, "GLiNER"
)

print("Regex baseline:", json.dumps(regex_summary, indent=2))
print("GLiNER:", json.dumps(gliner_summary, indent=2))


Regex baseline: {
  "method": "Regex baseline",
  "TP": 123,
  "FP": 6,
  "FN": 6,
  "precision": 0.9534883720930233,
  "recall": 0.9534883720930233,
  "f1": 0.9534883720930233
}
GLiNER: {
  "method": "GLiNER",
  "TP": 30,
  "FP": 29,
  "FN": 99,
  "precision": 0.5084745762711864,
  "recall": 0.23255813953488372,
  "f1": 0.3191489361702128
}


## 6. Error analysis

Review false positives and false negatives before interpreting scores. A legitimate vendor domain may be correctly extracted as a domain but should not be treated as a malicious IOC. CVEs are vulnerability references, not malicious infrastructure.


In [20]:
for row in regex_rows + gliner_rows:
    print("\nMETHOD:", row["method"], "| ADVISORY:", row["title"])
    print("TP:", row["true_positive"], "FP:", row["false_positive"], "FN:", row["false_negative"])
    print("Missed:", row["missed_entities"])
    print("Extra predictions (sample):", row["extra_entities_sample"])



METHOD: Regex baseline | ADVISORY: Multiple Vulnerabilities in Veeam Products
TP: 20 FP: 1 FN: 1
Missed: [('veeam', 'organization')]
Extra predictions (sample): [('cert-in.org.in', 'domain')]

METHOD: Regex baseline | ADVISORY: Preventing Online scams
TP: 0 FP: 1 FN: 2
Missed: [('lottery and prize scams', 'scam_category'), ('phishing scams', 'scam_category')]
Extra predictions (sample): [('cert-in.org.in', 'domain')]

METHOD: Regex baseline | ADVISORY: Multiple Vulnerabilities in Apple Products
TP: 89 FP: 1 FN: 1
Missed: [('apple', 'organization')]
Extra predictions (sample): [('cert-in.org.in', 'domain')]

METHOD: Regex baseline | ADVISORY: Command Injection Vulnerability in D-Link NAS
TP: 5 FP: 2 FN: 1
Missed: [('d-link', 'organization')]
Extra predictions (sample): [('12d6b683e67c80c49ffcc9214c239a07', 'md5'), ('cert-in.org.in', 'domain')]

METHOD: Regex baseline | ADVISORY: Multiple vulnerabilities in Web Management Interface of PAN-OS
TP: 9 FP: 1 FN: 1
Missed: [('palo alto networ

## 7. Evaluation summary and limitations

Include the actual metric outputs in the capstone report. Do not pre-fill or invent results.

### Limitations
1. The gold set is small and manually labelled; inter-annotator agreement was not measured.
2. The current gold set is not exhaustive. Unlabelled entities can appear as apparent false positives.
3. Exact-span evaluation can penalize reasonable boundary variations.
4. GLiNER confidence scores are not calibrated probabilities of maliciousness.
5. This notebook evaluates entity extraction, not threat attribution, maliciousness validation, or direct coverage of all four BFSI consortium priorities.
6. The regex baseline here is a transparent comparable baseline; it is not a byte-for-byte execution of every rule in the production collector.
7. All outputs must retain source provenance and must not be promoted to confirmed malicious indicators without validation.

### Reproducibility
Record the model name, threshold, package versions, gold-set changes, and date of execution when reporting results.


In [21]:

def hybrid_extract(text, threshold=0.3):
    """
    Combine regex and GLiNER extractions.
    Preserve provenance and deduplicate exact span/type matches.
    """
    combined = []

    # Structured entity extraction
    for entity in regex_extract(text):
        combined.append({
            **entity,
            "method": "regex",
        })

    # Contextual entity extraction
    for entity in gliner_extract(text, threshold=threshold):
        combined.append({
            **entity,
            "method": "gliner",
        })

    # Merge exact span/type matches while preserving both methods.
    merged = {}

    for entity in combined:
        key = (
            entity["start"],
            entity["end"],
            entity["label"],
            entity["text"].casefold(),
        )

        if key not in merged:
            merged[key] = {
                **entity,
                "methods": [entity["method"]],
            }
            merged[key].pop("method", None)
        else:
            if entity["method"] not in merged[key]["methods"]:
                merged[key]["methods"].append(entity["method"])

            # Retain GLiNER's confidence score when available.
            if "score" in entity:
                merged[key]["score"] = entity["score"]

    return list(merged.values())


# Demonstrate the hybrid extractor on the first advisory.
example_text = gold_examples[0]["text"]
hybrid_results = hybrid_extract(example_text)

print("Advisory:", gold_examples[0]["title"])
print("Hybrid entities:", len(hybrid_results))
print("\nSample results:")

for entity in hybrid_results[:15]:
    print({
        "text": entity["text"],
        "label": entity["label"],
        "methods": entity["methods"],
        "score": entity.get("score"),
    })


Advisory: Multiple Vulnerabilities in Veeam Products
Hybrid entities: 38

Sample results:
{'text': 'CVE-2024-40711', 'label': 'cve', 'methods': ['regex', 'gliner'], 'score': 0.973407506942749}
{'text': 'CVE-2024-39715', 'label': 'cve', 'methods': ['regex', 'gliner'], 'score': 0.9436139464378357}
{'text': 'CVE-2024-39714', 'label': 'cve', 'methods': ['regex', 'gliner'], 'score': 0.8859983682632446}
{'text': 'CVE-2024-39718', 'label': 'cve', 'methods': ['regex', 'gliner'], 'score': 0.7870323657989502}
{'text': 'CVE-2024-38650', 'label': 'cve', 'methods': ['regex', 'gliner'], 'score': 0.685935378074646}
{'text': 'CVE-2024-38651', 'label': 'cve', 'methods': ['regex', 'gliner'], 'score': 0.5636246800422668}
{'text': 'CVE-2024-40709', 'label': 'cve', 'methods': ['regex', 'gliner'], 'score': 0.6879600882530212}
{'text': 'CVE-2024-40710', 'label': 'cve', 'methods': ['regex', 'gliner'], 'score': 0.6683087944984436}
{'text': 'CVE-2024-40711', 'label': 'cve', 'methods': ['regex', 'gliner'], 'scor

In [22]:

from collections import Counter, defaultdict

# Count repeated logical entities by normalised value and type.
logical_entities = defaultdict(list)

for entity in hybrid_results:
    key = (entity["label"], entity["text"].casefold().strip())
    logical_entities[key].append(entity)

duplicates = {
    key: entries
    for key, entries in logical_entities.items()
    if len(entries) > 1
}

print("Total hybrid entities:", len(hybrid_results))
print("Unique entity type/value pairs:", len(logical_entities))
print("Repeated logical entities:", len(duplicates))

for (label, value), entries in duplicates.items():
    print(f"\n{label}: {value}")
    for entry in entries:
        print(
            "  methods =", entry["methods"],
            "| span =", (entry["start"], entry["end"]),
            "| score =", entry.get("score"),
        )


Total hybrid entities: 38
Unique entity type/value pairs: 28
Repeated logical entities: 6

cve: cve-2024-40711
  methods = ['regex', 'gliner'] | span = (206, 220) | score = 0.973407506942749
  methods = ['regex', 'gliner'] | span = (704, 718) | score = 0.6936576962471008

url: https://www.veeam.com/kb4649
  methods = ['regex'] | span = (461, 489) | score = None
  methods = ['regex'] | span = (515, 543) | score = None
  methods = ['regex'] | span = (561, 589) | score = None

domain: www.veeam.com
  methods = ['regex'] | span = (469, 482) | score = None
  methods = ['regex'] | span = (523, 536) | score = None
  methods = ['regex'] | span = (569, 582) | score = None

organization: veeam
  methods = ['gliner'] | span = (70, 75) | score = 0.8447592258453369
  methods = ['gliner'] | span = (232, 237) | score = 0.7002620100975037
  methods = ['gliner'] | span = (509, 514) | score = 0.8228726387023926
  methods = ['gliner'] | span = (555, 560) | score = 0.797225296497345

organization: ministr

In [23]:

from collections import defaultdict

deduplicated_entities = {}

for entity in hybrid_results:
    key = (
        entity["label"].casefold(),
        entity["text"].casefold().strip(),
    )

    if key not in deduplicated_entities:
        deduplicated_entities[key] = {
            "text": entity["text"],
            "label": entity["label"],
            "methods": set(entity["methods"]),
            "mention_count": 1,
            "scores": (
                [entity["score"]]
                if "score" in entity else []
            ),
        }
    else:
        item = deduplicated_entities[key]
        item["mention_count"] += 1
        item["methods"].update(entity["methods"])

        if "score" in entity:
            item["scores"].append(entity["score"])

# Convert sets to lists for readable output.
deduplicated_entities = [
    {
        **entity,
        "methods": sorted(entity["methods"]),
        "scores": entity["scores"],
    }
    for entity in deduplicated_entities.values()
]

print("Original hybrid entities:", len(hybrid_results))
print("Deduplicated entities:", len(deduplicated_entities))
print("\nSample deduplicated entities:")

for entity in deduplicated_entities[:15]:
    print(entity)


Original hybrid entities: 38
Deduplicated entities: 28

Sample deduplicated entities:
{'text': 'CVE-2024-40711', 'label': 'cve', 'methods': ['gliner', 'regex'], 'mention_count': 2, 'scores': [0.973407506942749, 0.6936576962471008]}
{'text': 'CVE-2024-39715', 'label': 'cve', 'methods': ['gliner', 'regex'], 'mention_count': 1, 'scores': [0.9436139464378357]}
{'text': 'CVE-2024-39714', 'label': 'cve', 'methods': ['gliner', 'regex'], 'mention_count': 1, 'scores': [0.8859983682632446]}
{'text': 'CVE-2024-39718', 'label': 'cve', 'methods': ['gliner', 'regex'], 'mention_count': 1, 'scores': [0.7870323657989502]}
{'text': 'CVE-2024-38650', 'label': 'cve', 'methods': ['gliner', 'regex'], 'mention_count': 1, 'scores': [0.685935378074646]}
{'text': 'CVE-2024-38651', 'label': 'cve', 'methods': ['gliner', 'regex'], 'mention_count': 1, 'scores': [0.5636246800422668]}
{'text': 'CVE-2024-40709', 'label': 'cve', 'methods': ['gliner', 'regex'], 'mention_count': 1, 'scores': [0.6879600882530212]}
{'text'

In [24]:

hybrid_summary = []

for example in gold_examples:
    entities = hybrid_extract(example["text"])

    unique_keys = {
        (
            entity["label"].casefold(),
            entity["text"].casefold().strip(),
        )
        for entity in entities
    }

    method_counts = {"regex": 0, "gliner": 0, "both": 0}

    # Count unique entities by extraction method.
    unique_methods = {}

    for entity in entities:
        key = (
            entity["label"].casefold(),
            entity["text"].casefold().strip(),
        )
        unique_methods.setdefault(key, set()).update(entity["methods"])

    for methods in unique_methods.values():
        if methods == {"regex", "gliner"}:
            method_counts["both"] += 1
        else:
            for method in methods:
                method_counts[method] += 1

    hybrid_summary.append({
        "advisory": example["title"],
        "raw_mentions": len(entities),
        "unique_entities": len(unique_keys),
        **method_counts,
    })

for row in hybrid_summary:
    print(row)

print("\nTotal unique entities across advisory-level results:",
      sum(row["unique_entities"] for row in hybrid_summary))


{'advisory': 'Multiple Vulnerabilities in Veeam Products', 'raw_mentions': 38, 'unique_entities': 28, 'regex': 3, 'gliner': 7, 'both': 18}
{'advisory': 'Preventing Online scams', 'raw_mentions': 18, 'unique_entities': 15, 'regex': 1, 'gliner': 14, 'both': 0}
{'advisory': 'Multiple Vulnerabilities in Apple Products', 'raw_mentions': 138, 'unique_entities': 92, 'regex': 87, 'gliner': 2, 'both': 3}
{'advisory': 'Command Injection Vulnerability in D-Link NAS', 'raw_mentions': 19, 'unique_entities': 14, 'regex': 6, 'gliner': 7, 'both': 1}
{'advisory': 'Multiple vulnerabilities in Web Management Interface of PAN-OS', 'raw_mentions': 36, 'unique_entities': 15, 'regex': 8, 'gliner': 5, 'both': 2}

Total unique entities across advisory-level results: 164
